# Math 01: Algebra — Linear Equations**Learning Objectives** — By the end of this notebook you will be able to:1. Define what a linear equation is and identify its components2. Solve single-variable linear equations using inverse operations3. Recognise the three solution cases (unique, none, infinitely many)4. Translate algebraic solutions into Python code from scratch5. Verify solutions computationally and handle pathological edge cases**Prerequisites:** Basic Python (variables, functions, if/else)  **Estimated time:** 90 minutes  **Textbook reference:** *Justin Skycak — Algebra*, Part 1 §1.1–1.5

In [ ]:
# Google Colab Setup!pip install -q ipywidgets jupyterquiz ipytest plotly sympy networkx pandas numpy matplotlib tqdmimport json, os, sysfrom pathlib import Pathif Path('/content').exists():    repo_root = Path('/content/upskill')else:    repo_root = Path.cwd()if str(repo_root) not in sys.path:    sys.path.insert(0, str(repo_root))try:    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )except ModuleNotFoundError:    Path('learning_tools.py').write_text('"""Colab-first learning helpers for the interactive math curriculum.\n\nThe functions in this module are intentionally lightweight: they work in a\nplain notebook, in Google Colab, and in local Jupyter without requiring a\nbackend service.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, asdict, field\nfrom datetime import datetime, timedelta, timezone\nimport json\nimport math\nimport os\nimport re\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable\n\n\nSCHEMA_VERSION = 1\nDEFAULT_PROGRESS_FILENAME = "upskill_math_progress.json"\nDRIVE_PROGRESS_DIR = "MyDrive/upskill"\n\n\ndef _utcnow() -> datetime:\n    return datetime.now(timezone.utc)\n\n\ndef _iso(dt: datetime) -> str:\n    return dt.astimezone(timezone.utc).replace(microsecond=0).isoformat()\n\n\ndef _parse_iso(value: str | None) -> datetime | None:\n    if not value:\n        return None\n    try:\n        return datetime.fromisoformat(value.replace("Z", "+00:00"))\n    except ValueError:\n        return None\n\n\n@dataclass\nclass Skill:\n    """A small curriculum skill descriptor."""\n\n    id: str\n    title: str\n    notebook: str\n    level: int\n    prerequisites: list[str] = field(default_factory=list)\n    tags: list[str] = field(default_factory=list)\n\n    def to_dict(self) -> dict[str, Any]:\n        return asdict(self)\n\n\ndef setup_colab(\n    progress_filename: str = DEFAULT_PROGRESS_FILENAME,\n    mount_drive: bool = True,\n    verbose: bool = True,\n) -> Path:\n    """Prepare a notebook session and return the progress-file path.\n\n    In Colab, this mounts Google Drive when available. Outside Colab, it falls\n    back to the current working directory.\n    """\n\n    progress_path = Path(progress_filename)\n    try:\n        import google.colab  # type: ignore  # noqa: F401\n        in_colab = True\n    except Exception:\n        in_colab = False\n\n    if in_colab and mount_drive:\n        try:\n            from google.colab import drive  # type: ignore\n\n            drive.mount("/content/drive")\n            drive_dir = Path("/content/drive") / DRIVE_PROGRESS_DIR\n            drive_dir.mkdir(parents=True, exist_ok=True)\n            progress_path = drive_dir / progress_filename\n        except Exception as exc:\n            if verbose:\n                print(f"Drive mount skipped; using local progress file. ({exc})")\n\n    if verbose:\n        print(f"Learning environment ready. Progress: {progress_path}")\n    return progress_path\n\n\nclass ProgressStore:\n    """JSON-backed spaced-repetition progress store."""\n\n    def __init__(self, path: str | Path | None = None, learner_id: str = "local"):\n        self.path = Path(path or DEFAULT_PROGRESS_FILENAME)\n        self.learner_id = learner_id\n        self.data = self._load()\n\n    def _default_data(self) -> dict[str, Any]:\n        return {\n            "schema_version": SCHEMA_VERSION,\n            "learner_id": self.learner_id,\n            "skills": {},\n        }\n\n    def _load(self) -> dict[str, Any]:\n        if not self.path.exists():\n            return self._default_data()\n        try:\n            data = json.loads(self.path.read_text(encoding="utf-8"))\n        except Exception:\n            return self._default_data()\n        if data.get("schema_version") != SCHEMA_VERSION:\n            data["schema_version"] = SCHEMA_VERSION\n        data.setdefault("learner_id", self.learner_id)\n        data.setdefault("skills", {})\n        return data\n\n    def save(self) -> None:\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.path.write_text(json.dumps(self.data, indent=2), encoding="utf-8")\n\n    def get(self, skill_id: str) -> dict[str, Any]:\n        return self.data.setdefault("skills", {}).setdefault(skill_id, {\n            "attempts": 0,\n            "correct": 0,\n            "confidence": 0,\n            "mastery": 0.0,\n            "last_seen": None,\n            "next_review": None,\n        })\n\n    def record_attempt(\n        self,\n        skill_id: str,\n        correct: bool,\n        confidence: int = 3,\n        item_type: str = "practice",\n    ) -> dict[str, Any]:\n        confidence = max(1, min(5, int(confidence)))\n        row = self.get(skill_id)\n        attempts = int(row.get("attempts", 0)) + 1\n        correct_count = int(row.get("correct", 0)) + int(bool(correct))\n        accuracy = correct_count / attempts\n        confidence_factor = confidence / 5\n        mastery = round((0.75 * accuracy) + (0.25 * confidence_factor), 3)\n\n        row.update({\n            "attempts": attempts,\n            "correct": correct_count,\n            "confidence": confidence,\n            "mastery": mastery,\n            "last_seen": _iso(_utcnow()),\n            "last_item_type": item_type,\n            "next_review": _iso(_utcnow() + review_interval(correct, confidence, attempts)),\n        })\n        self.save()\n        return row\n\n    def review_queue(\n        self,\n        skills: Iterable[Skill | dict[str, Any]] | None = None,\n        limit: int = 8,\n        now: datetime | None = None,\n    ) -> list[dict[str, Any]]:\n        now = now or _utcnow()\n        known_titles = {}\n        if skills:\n            for skill in skills:\n                item = skill.to_dict() if isinstance(skill, Skill) else skill\n                known_titles[item["id"]] = item\n\n        due: list[dict[str, Any]] = []\n        for skill_id, row in self.data.get("skills", {}).items():\n            next_review = _parse_iso(row.get("next_review"))\n            if next_review is None or next_review <= now:\n                due.append({\n                    "id": skill_id,\n                    "title": known_titles.get(skill_id, {}).get("title", skill_id),\n                    "mastery": row.get("mastery", 0.0),\n                    "next_review": row.get("next_review"),\n                    "attempts": row.get("attempts", 0),\n                })\n        due.sort(key=lambda item: (item["mastery"], item["next_review"] or ""))\n        return due[:limit]\n\n    def weak_skills(self, limit: int = 8) -> list[tuple[str, dict[str, Any]]]:\n        rows = list(self.data.get("skills", {}).items())\n        rows.sort(key=lambda kv: (kv[1].get("mastery", 0.0), -kv[1].get("attempts", 0)))\n        return rows[:limit]\n\n\ndef review_interval(correct: bool, confidence: int, attempts: int = 1) -> timedelta:\n    """Return the next review interval from correctness and confidence."""\n\n    if not correct:\n        return timedelta(hours=6) if attempts <= 1 else timedelta(days=1)\n    if confidence <= 2:\n        return timedelta(days=1)\n    if confidence == 3:\n        return timedelta(days=3)\n    high_confidence_days = [7, 14, 30, 60]\n    return timedelta(days=high_confidence_days[min(max(attempts - 1, 0), len(high_confidence_days) - 1)])\n\n\ndef record_attempt(\n    skill_id: str,\n    correct: bool,\n    confidence: int = 3,\n    item_type: str = "practice",\n    store: ProgressStore | None = None,\n) -> dict[str, Any]:\n    store = store or ProgressStore()\n    return store.record_attempt(skill_id, correct, confidence, item_type)\n\n\ndef check(\n    name: str,\n    actual: Any,\n    expected: Any,\n    tolerance: float | None = None,\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Friendly equality/numeric check with optional progress recording."""\n\n    if tolerance is not None and isinstance(actual, (int, float)) and isinstance(expected, (int, float)):\n        ok = math.isclose(actual, expected, rel_tol=tolerance, abs_tol=tolerance)\n    else:\n        ok = actual == expected\n\n    if ok:\n        print(f"PASS: {name}")\n    else:\n        print(f"TRY AGAIN: {name}")\n        print(f"  expected: {expected!r}")\n        print(f"  actual:   {actual!r}")\n\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "check", store)\n    return ok\n\n\ndef code_check(\n    name: str,\n    fn: Callable[..., Any],\n    cases: Iterable[tuple[tuple[Any, ...], Any] | tuple[tuple[Any, ...], dict[str, Any], Any]],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Run multiple function cases and report the first failure."""\n\n    all_ok = True\n    for index, case in enumerate(cases, 1):\n        if len(case) == 2:\n            args, expected = case  # type: ignore[misc]\n            kwargs = {}\n        else:\n            args, kwargs, expected = case  # type: ignore[misc]\n        try:\n            actual = fn(*args, **kwargs)\n            ok = actual == expected\n        except Exception as exc:\n            actual = f"{type(exc).__name__}: {exc}"\n            ok = False\n        if not ok:\n            all_ok = False\n            print(f"TRY AGAIN: {name} case {index}")\n            print(f"  args:     {args}")\n            print(f"  expected: {expected!r}")\n            print(f"  actual:   {actual!r}")\n            break\n    if all_ok:\n        print(f"PASS: {name} ({index} cases)")\n    if skill_id:\n        record_attempt(skill_id, all_ok, confidence, "code_check", store)\n    return all_ok\n\n\ndef normalize_answer(value: Any) -> str:\n    text = str(value).strip().lower()\n    text = re.sub(r"\\s+", " ", text)\n    text = re.sub(r"[^a-z0-9.+/=\\- ]", "", text)\n    text = re.sub(r"\\s*=\\s*", "=", text)\n    return text\n\n\ndef short_answer_check(\n    name: str,\n    answer: Any,\n    accepted: Iterable[Any],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    normalized = normalize_answer(answer)\n    accepted_norm = {normalize_answer(item) for item in accepted}\n    ok = normalized in accepted_norm\n    print(f"{\'PASS\' if ok else \'TRY AGAIN\'}: {name}")\n    if not ok:\n        print("  Re-read the prompt and try to retrieve the answer before opening a hint.")\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "short_answer", store)\n    return ok\n\n\ndef hint_box(title: str, hints: list[str], unlock: bool = False) -> None:\n    """Display staged hints. Full hints require unlock=True."""\n\n    print(title)\n    if not hints:\n        print("No hints for this item.")\n        return\n    if not unlock:\n        print(f"Hint 1: {hints[0]}")\n        print("Run again with unlock=True after a real attempt to see more.")\n        return\n    for index, hint in enumerate(hints, 1):\n        print(f"Hint {index}: {hint}")\n\n\ndef mastery_dashboard(\n    store: ProgressStore | None = None,\n    skills: Iterable[Skill | dict[str, Any]] | None = None,\n    next_notebook: str | None = None,\n) -> None:\n    store = store or ProgressStore()\n    due = store.review_queue(skills=skills)\n    weak = store.weak_skills()\n    total = len(store.data.get("skills", {}))\n    mastered = sum(1 for row in store.data.get("skills", {}).values() if row.get("mastery", 0) >= 0.8)\n\n    print("Mastery Dashboard")\n    print("=================")\n    print(f"Skills attempted: {total}")\n    print(f"Skills at mastery >= 0.80: {mastered}")\n    if due:\n        print("\\nDue for review:")\n        for item in due:\n            print(f"- {item[\'title\']} (mastery {item[\'mastery\']})")\n    else:\n        print("\\nNo due reviews yet.")\n    if weak:\n        print("\\nWeakest skills:")\n        for skill_id, row in weak:\n            print(f"- {skill_id}: mastery {row.get(\'mastery\', 0)} after {row.get(\'attempts\', 0)} attempts")\n    if next_notebook:\n        print(f"\\nRecommended next notebook: {next_notebook}")\n', encoding='utf-8')    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )progress_path = setup_colab()store = ProgressStore(progress_path)print('Ready for retrieval practice.')

In [ ]:
NOTEBOOK = {    "id": "math_01_algebra_linear_equations",    "level": 1,    "title": "Math 01: Algebra \u2014 Linear Equations",    "prerequisites": [        "Basic Python (variables",        "functions",        "if/else)"    ],    "skills_taught": [        "lesson.math_01_algebra_linear_equations"    ],    "skills_practiced": [        "py.arithmetic.expressions"    ],    "next_notebook": "math_02_calculus_derivatives.ipynb"}SKILLS = [    {        "id": "lesson.math_01_algebra_linear_equations",        "title": "Lesson Math 01 Algebra Linear Equations",        "notebook": "math_01_algebra_linear_equations.ipynb",        "level": 1    },    {        "id": "py.arithmetic.expressions",        "title": "Py Arithmetic Expressions",        "notebook": "math_01_algebra_linear_equations.ipynb",        "level": 1    }]

## Before You Learn: Retrieval GateClose notes for a moment. Try to pull prerequisite ideas from memory before continuing. Getting stuck is useful data for the spaced-review system.

In [ ]:
due = store.review_queue(skills=SKILLS, limit=5)if due:    print('Due review items:')    for item in due:        print(f"- {item['title']} (mastery {item['mastery']})")else:    print('No due reviews yet. Use the recall prompt below to warm up.')

**Recall prompt.** Before reading, name one key idea or procedure you remember from: Basic Python (variables, functions, if/else).

In [ ]:
recall_answer = ''  # write a one-sentence memory before continuingretrieved = len(recall_answer.strip()) >= 12record = store.record_attempt('lesson.math_01_algebra_linear_equations', retrieved, confidence=3, item_type='prerequisite_recall')print('Recorded retrieval attempt.' if retrieved else 'Write a real memory first, then rerun this cell.')record

## Learning LoopUse the existing lesson cells below as the micro-lesson and practice surface. For each exercise: predict first, attempt from memory, run checks, then open explanations only after the attempt.

In [ ]:
# Google Colab Setup!pip install -q sympy matplotlib plotly ipywidgetsimport sympy as sp; sp.init_printing()import numpy as np, matplotlib.pyplot as pltprint('Ready ✅')

---## Table of Contents1. [Phase −1: Theoretical Foundation](#phase-1)  2. [Phase 0: Math Foundation Practice](#phase-0)  3. [Phase 1: Algorithm Construction](#phase-1-algo)  4. [Phase 2: Experimental Verification](#phase-2)  5. [Phase 3: Olympiad Extension](#phase-3)

---<a id='phase-1'></a>## Phase −1 — Theoretical Foundation (Kiselev / Gelfand Rigor)### What is a linear equation?A **linear equation** is an equality containing only addition, subtraction,multiplication and division — no exponents, roots, or transcendentals.$$ax + b = c \qquad (a,\,b,\,c \in \mathbb{R},\; a \neq 0)$$**Axiom (Additive & Multiplicative Inverse):** For every real number $r$,there exists $-r$ such that $r + (-r) = 0$, and for $r\neq 0$ there exists$r^{-1}$ such that $r \cdot r^{-1} = 1$.These two axioms are the *entire engine* behind solving any linear equation:we apply inverse operations to both sides until the variable is isolated.

### Guided Proof — Uniqueness of the Solution**Theorem.** If $a \neq 0$, the equation $ax + b = c$ has exactly one solution $x = \frac{c-b}{a}$.*Proof.*1. Subtract $b$ from both sides (additive inverse): $ax = c - b$2. Divide both sides by $a$ (multiplicative inverse, valid since $a\neq 0$): $x = \frac{c-b}{a}$3. Uniqueness: suppose $x_1,x_2$ both satisfy the equation.     Then $ax_1 + b = c = ax_2 + b \Rightarrow ax_1 = ax_2 \Rightarrow x_1 = x_2$. $\square$

In [ ]:
# Verify the proof symbolically with SymPyimport sympy as spx, a, b, c = sp.symbols('x a b c')equation = sp.Eq(a*x + b, c)solution = sp.solve(equation, x)print('Symbolic solution:', solution)  # [(c - b)/a]# Verify: substitute backcheck = (a * solution[0] + b).simplify()print('Substitution gives:', check, '== c ?', sp.simplify(check - c) == 0)

### The Three Cases| Case | Condition | What happens algebraically | Example ||---|---|---|---|| **Unique solution** | $a \neq 0$ | Variable isolates to one value | $2x+3=7 \Rightarrow x=2$ || **No solution** | $a=0,\; b \neq c$ | Variable vanishes, leaving a false statement | $0x+3=7 \Rightarrow 3=7$ ✗ || **Infinitely many** | $a=0,\; b=c$ | Variable vanishes, leaving a true statement | $0x+5=5 \Rightarrow 5=5$ ✓ |> 📖 *From Algebra §1.1:* 'Most linear equations have a single solution. We find it by> performing operations on both sides to isolate the variable.'

### Comprehension Check ✅Before continuing, answer in your head:1. Why do we need $a \neq 0$ for a unique solution?2. Give an example of a linear equation with no solution.3. What does "infinitely many solutions" look like geometrically on a number line?<details><summary>💡 Hints</summary>1. If $a=0$, dividing by $a$ is undefined — we lose the variable entirely.  2. $5x + 3 = 5x + 7$ → subtract $5x$: $3 = 7$ (false).  3. Every real number satisfies the equation — the "solution set" is the whole number line.</details>

---<a id='phase-0'></a>## Phase 0 — Math Foundation Practice### Worked Example: Solving a Multi-Step EquationSolve $3(x + 2) - 4 = 11$ step by step.| Step | Operation | Result ||------|-----------|--------|| Start | — | $3(x+2) - 4 = 11$ || Distribute | $3 \cdot x + 3 \cdot 2$ | $3x + 6 - 4 = 11$ || Simplify | $6 - 4 = 2$ | $3x + 2 = 11$ || Subtract 2 | additive inverse | $3x = 9$ || Divide by 3 | multiplicative inverse | $x = 3$ |**Verification:** $3(3+2)-4 = 3(5)-4 = 15-4 = 11$ ✓

In [ ]:
# Worked example — run this cell to see the step-by-step solutiona_val, b_val, c_val = 3, 2, 11  # 3(x+2) - 4 = 11 → 3x + 6 - 4 = 11 → 3x + 2 = 11# Rearranged: a_eff*x + b_eff = c  where a_eff=3, b_eff=2, c=11x_solution = (c_val - b_val) / a_valprint(f'Solution: x = ({c_val} - {b_val}) / {a_val} = {x_solution}')print(f'Check: 3*({x_solution}+2) - 4 = {3*(x_solution+2)-4}')

### 🎯 Your Turn — Solve by Hand, then Verify in CodeSolve each equation on paper first, then fill in the function below to check computationally.1. $5x - 7 = 18$2. $-2x + 10 = 4$3. $4(x - 3) + 2 = 14$

In [ ]:
def solve_linear(a, b, c):    """Solve ax + b = c for x. Return the solution or raise if a == 0."""    # YOUR CODE HERE    pass# Tests — uncomment after implementing:# assert solve_linear(5, -7, 18) == 5.0,  'Problem 1 failed'# assert solve_linear(-2, 10, 4) == 3.0,  'Problem 2 failed'# assert solve_linear(4, -10, 14) == 6.0, 'Problem 3 (after expanding) failed'# print('All Phase 0 tests passed ✅')

<details><summary>💡 Hint</summary>The formula is $x = \frac{c - b}{a}$. But what should you do when $a = 0$?Consider raising a `ValueError`.</details>

### Common Mistakes ⚠️1. **Forgetting to distribute** — $3(x+2) \neq 3x + 2$, it's $3x + 6$2. **Sign errors with negatives** — when subtracting a negative: $x - (-3) = x + 3$3. **Dividing before subtracting** — always undo addition/subtraction first, *then* multiplication4. **Ignoring the $a=0$ case** — your code should handle this gracefully

---<a id='phase-1-algo'></a>## Phase 1 — Micro-Scaffolded Algorithm ConstructionNow we build a **complete linear equation solver** from scratch — no libraries.This solver is the foundation for cryptography breaking in Notebook 07.### Step 1: Basic SolverWrite `solve_linear(a, b, c)` that handles all three cases.

In [ ]:
def solve_linear(a, b, c):    """    Solve ax + b = c for x.        Returns:        float: the unique solution if a != 0    Raises:        ValueError: if a == 0 (no solution or infinitely many)    """    # YOUR CODE HERE    pass

In [ ]:
# ── Step 1 Verification ──# Uncomment after implementing:# assert solve_linear(2, 3, 7) == 2.0# assert solve_linear(1, 0, 5) == 5.0# assert solve_linear(-3, 9, 0) == 3.0# try:#     solve_linear(0, 3, 7)#     assert False, 'Should have raised ValueError'# except ValueError:#     pass# print('Step 1 passed ✅')

### Step 2: Multi-Step SolverSolve $a(x + b) + c = d$ for $x$.*Strategy:* Expand → rearrange into standard $Ax + B = C$ form → call `solve_linear`.

In [ ]:
def solve_complex_linear(a, b, c, d):    """    Solve a(x + b) + c = d for x.    Expand: ax + ab + c = d  →  ax + (ab+c) = d    """    # YOUR CODE HERE — hint: reuse solve_linear!    pass

In [ ]:
# ── Step 2 Verification ──# assert solve_complex_linear(3, 2, -4, 11) == 3.0# assert solve_complex_linear(2, -1, 5, 9) == 3.0# print('Step 2 passed ✅')

### Step 3: Batch Verifier (Interleaved Practice)Write a function that takes a list of $(a,b,c)$ triples,solves each one, plugs the answer back in, and reports whether each is correct.*This revisits loops and list-building from Notebook 01.*

In [ ]:
def verify_batch(problems):    """    problems: list of (a, b, c) tuples for ax + b = c    Returns: list of (x, is_correct) tuples    """    results = []    for a, b, c in problems:        # YOUR CODE HERE        pass    return results# Test:# problems = [(2,3,7), (5,-7,18), (-2,10,4), (1,0,0)]# for (x, ok) in verify_batch(problems):#     print(f'x={x:.2f}  correct={ok}')

### Step 4: Spaced Repetition — Slope-Intercept Connection> 📖 *From Algebra §1.2:* A two-variable linear equation $y = mx + b$ graphs as a straight line.> The slope $m$ tells how steep it is; the y-intercept $b$ is where it crosses the y-axis.The solution to $ax + b = c$ is geometrically the **x-coordinate where the line $y = ax + b$crosses the horizontal line $y = c$**.Write a function that plots this geometric interpretation.

In [ ]:
def plot_linear_solution(a, b, c):    """Plot y = ax + b and y = c, marking their intersection."""    import matplotlib.pyplot as plt    import numpy as np        x_sol = (c - b) / a    x_range = np.linspace(x_sol - 5, x_sol + 5, 200)        fig, ax_plot = plt.subplots(figsize=(8, 5))    ax_plot.plot(x_range, a * x_range + b, 'b-', linewidth=2, label=f'y = {a}x + {b}')    ax_plot.axhline(y=c, color='r', linestyle='--', linewidth=1.5, label=f'y = {c}')    ax_plot.plot(x_sol, c, 'ko', markersize=10, zorder=5)    ax_plot.annotate(f'  x = {x_sol:.2f}', (x_sol, c), fontsize=12, fontweight='bold')    ax_plot.set_xlabel('x'); ax_plot.set_ylabel('y')    ax_plot.set_title(f'Geometric Solution of {a}x + {b} = {c}')    ax_plot.legend(); ax_plot.grid(True, alpha=0.3)    plt.tight_layout(); plt.show()# Run it:plot_linear_solution(2, 3, 7)

---<a id='phase-2'></a>## Phase 2 — Experimental Verification & Visualization### What We're TestingOur `solve_linear` function uses exact arithmetic: $x = (c-b)/a$.But floating-point numbers on a computer are *approximations*.Can we find inputs where floating-point arithmetic gives the **wrong** answer?

In [ ]:
# ── Experiment 1: Floating-point precision limits ──# Try very large and very small coefficientsimport numpy as nptest_cases = [    (1e15, 1e15, 2e15,   'large coefficients'),    (1e-15, 1e-15, 2e-15, 'tiny coefficients'),    (1e15, -1e15, 1,      'catastrophic cancellation'),    (1e-300, 0, 1e-300,   'near underflow'),]for a, b, c, label in test_cases:    x = (c - b) / a    residual = abs(a * x + b - c)    print(f'{label:30s}  x={x:20.6e}  residual={residual:.2e}')

In [ ]:
# ── Experiment 2: The Pathological Case ──# When (c - b) suffers catastrophic cancellationa = 1.0b = 1e16c = 1e16 + 1  # True answer: x = 1x_computed = (c - b) / aprint(f'Expected: x = 1.0')print(f'Computed: x = {x_computed}')print(f'Error:    {abs(x_computed - 1.0)}')print()print('⚠️  When b and c are very close in magnitude but huge,')print('    (c - b) loses precision due to catastrophic cancellation.')

In [ ]:
# ── Visualization: Error growth ──import matplotlib.pyplot as pltimport numpy as npscales = np.logspace(0, 16, 100)errors = []for s in scales:    b_val = s    c_val = s + 1.0  # true x = 1.0    x_comp = (c_val - b_val) / 1.0    errors.append(abs(x_comp - 1.0))plt.figure(figsize=(9, 5))plt.loglog(scales, errors, 'r-', linewidth=2)plt.xlabel('Scale of b and c', fontsize=12)plt.ylabel('Absolute Error in x', fontsize=12)plt.title('Catastrophic Cancellation in Linear Equation Solving', fontsize=13)plt.grid(True, alpha=0.3)plt.tight_layout()plt.show()

### 🔍 Reflection1. At what scale does the error become non-negligible?2. Why does `(c - b)` lose precision when both are ≈ $10^{16}$?3. Could using `decimal.Decimal` or `fractions.Fraction` help?<details><summary>💡 Explanation after a retrieval attempt</summary>IEEE 754 `float64` has ~15–17 significant digits. When $b \approx c \approx 10^{16}$,the difference $c - b = 1$ occupies the 17th digit — beyond precision.Using `Fraction` or `Decimal` with sufficient precision eliminates this.</details>

---<a id='phase-3'></a>## Phase 3 — Olympiad Extension & Chalkboard Defense### Olympiad Challenge: Exact-Arithmetic SolverBuild a version of `solve_linear` that uses Python's `fractions.Fraction`to return **exact** rational answers, immune to floating-point errors.Then solve this system that would break a float-based solver:$$\frac{1}{3}x + \frac{1}{7} = \frac{10}{21}$$The exact answer is $x = 1$. Does your float solver agree?

In [ ]:
# ── Olympiad Extension: Exact Solver ──# Build this WITHOUT any scaffolding.from fractions import Fractiondef solve_linear_exact(a, b, c):    """Solve ax + b = c using exact rational arithmetic."""    # YOUR CODE HERE    pass# Test: solve (1/3)x + 1/7 = 10/21# x = solve_linear_exact(Fraction(1,3), Fraction(1,7), Fraction(10,21))# print(f'Exact: x = {x}')# print(f'Float: x = {(10/21 - 1/7) / (1/3)}')  # may not be exactly 1.0

### Chalkboard DefenseWrite a formal defense addressing:1. **Correctness:** Prove that `solve_linear(a,b,c)` returns the unique root of $ax+b=c$ for all $a\neq 0$.2. **Time Complexity:** What is the Big-O of your solver? (Hint: it's $O(1)$ — why?)3. **Numerical Stability:** Under what conditions does the floating-point solver fail,   and how does the `Fraction`-based solver fix this?*Write your defense below using LaTeX:*

**Defense:***(Your formal mathematical defense here)*<details><summary>💡 Example defense structure</summary>**Claim 1 (Correctness).** For $a \neq 0$, `solve_linear(a,b,c)` returns $x^* = (c-b)/a$.Substituting: $a \cdot \frac{c-b}{a} + b = (c-b) + b = c$. ✓**Claim 2 (Complexity).** The function performs 1 subtraction and 1 division: $O(1)$ time.**Claim 3 (Stability).** When $|b| \approx |c| \gg 1$, the subtraction $c - b$ sufferscatastrophic cancellation. The `Fraction` solver avoids this because it performsexact integer arithmetic on numerators and denominators.</details>

---### 📚 Further Reading- *Algebra* §1.2–1.5: Slope-intercept form, point-slope form, linear systems- *The Math Academy Way* Ch. 14: Minimizing Cognitive Load & Micro-Scaffolding- **Next notebook:** Math 02 (Calculus — Derivatives) builds on algebraic manipulation

## End-of-Notebook ReviewRetrieve the core idea one more time before leaving. This final recall makes the next review easier.

In [ ]:
final_recall = ''  # summarize the most important idea in your own wordsconfidence = 3  # set 1-5 after answeringok = len(final_recall.strip()) >= 20store.record_attempt('lesson.math_01_algebra_linear_equations', ok, confidence=confidence, item_type='end_review')mastery_dashboard(store=store, skills=SKILLS, next_notebook='math_02_calculus_derivatives.ipynb')